# PCA Visualization of Grid Random-Walk Embeddings

Pipeline (following the **Grid, fixed window 50** protocol):

1. Choose embedding field (`layer_26`).
2. Load one batch: **B = 16** files, one per grid node / random-walk start.
3. Each file already contains the **Nw = 50** recent-window slice.
4. Pool embeddings **by token id** across all 16 windows.
5. Compute per-token mean embedding **h_τ**.
6. Stack means into **H ∈ ℝ^{n_nodes × d}**.
7. PCA → **Z ∈ ℝ^{n_nodes × 2}**.
8. Scatter plot PC1 vs PC2, labelled by word.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from pathlib import Path

## Configuration

In [ ]:
ROOT = Path('/Users/giaco/Documents/projects/NLP/hico')

In [ ]:



# Token-id → word mapping (LLaMA-3 tokeniser ids for the 16 grid words)
ID_TO_WORD = {
    7160:  'sun',
    2082:  'code',
    14403: 'milk',
    12224: 'bird',
    9462:  'sand',
    1841:  'car',
    19151: 'egg',
    70368: 'mango',
    7033:  'math',
    43516: 'opera',
    3830:  'box',
    3838:  'house',
    24149: 'apple',
    11277: 'plane',
    4641:  'phone',
    7091:  'rock',
}

# Grid layout (from grid_layout.txt) – used to colour by row
GRID = [
    ['sun',   'code',  'milk',  'bird'],
    ['sand',  'car',   'egg',   'mango'],
    ['math',  'opera', 'box',   'house'],
    ['apple', 'plane', 'phone', 'rock'],
]

# Build word → (row, col) position on the grid
WORD_TO_POS = {}
for r, row in enumerate(GRID):
    for c, w in enumerate(row):
        WORD_TO_POS[w] = (r, c)

## Step 1–2: Load the batch (B = 16 sequences, each with Nw = 50 timesteps)

In [ ]:
EMB_DIR = ROOT / 'embeddings'

# Batch / grid parameters
B = 16              # batch size = number of grid nodes
NW = 50             # fixed recent window length
EMB_KEY = 'embeddings_last'   # embedding field inside each .pt
IDS_KEY = 'input_ids_last'    # token-id field inside each .pt

# File pattern: one file per batch element
FILE_PATTERN = 'reprs_paper_1400_16_b{batch_idx:04d}_layer26.pt'
all_token_ids = []   # will hold (B * Nw,) concatenated token ids
all_embeddings = []  # will hold (B * Nw, d) concatenated embeddings

for b in range(B):
    fpath = EMB_DIR / FILE_PATTERN.format(batch_idx=b)
    assert fpath.exists(), f'Missing file: {fpath}'

    save_obj = torch.load(fpath, map_location='cpu', weights_only=False)
    ids = save_obj[IDS_KEY]       # (Nw,)
    emb = save_obj[EMB_KEY]       # (Nw, d)

    # Normalise shapes (squeeze leading dim if present)
    if ids.ndim == 2 and ids.shape[0] == 1:
        ids = ids[0]
    if emb.ndim == 3 and emb.shape[0] == 1:
        emb = emb[0]

    # Take the recent window W_b = min(NW, L_b)
    L_b = ids.shape[0]
    W_b = min(NW, L_b)
    ids = ids[-W_b:]
    emb = emb[-W_b:]

    all_token_ids.append(ids)
    all_embeddings.append(emb.float())  # upcast fp16 → fp32 for numerics

all_token_ids = torch.cat(all_token_ids, dim=0)      # (B * Nw,)
all_embeddings = torch.cat(all_embeddings, dim=0)     # (B * Nw, d)

d = all_embeddings.shape[1]
print(f'Loaded batch: B={B}, window per sequence={NW}')
print(f'Total timesteps pooled: {all_embeddings.shape[0]}')
print(f'Embedding dim d={d}')
print(f'Unique token ids: {torch.unique(all_token_ids).shape[0]}')

## Steps 3–5: Pool by token id & compute mean embeddings

In [ ]:
unique_ids = sorted(ID_TO_WORD.keys())  # fixed sorted order for H matrix
n_nodes = len(unique_ids)

H = np.zeros((n_nodes, d), dtype=np.float32)
counts = {}  # m_τ for each token

ids_np = all_token_ids.numpy()
emb_np = all_embeddings.numpy()

for i, tau in enumerate(unique_ids):
    mask = ids_np == tau
    S_tau = emb_np[mask]               # all occurrences of token τ
    m_tau = S_tau.shape[0]
    counts[tau] = m_tau

    assert m_tau > 0, f'Coverage check failed: token {tau} ({ID_TO_WORD[tau]}) has 0 occurrences.'
    H[i] = S_tau.mean(axis=0)          # h_τ = mean embedding

print(f'H shape: {H.shape}  (n_nodes={n_nodes}, d={d})')
print()
print('Per-token occurrence counts m_τ:')
for tau in unique_ids:
    print(f'  {ID_TO_WORD[tau]:>8s} (id {tau:>6d}): m_τ = {counts[tau]}')

## Step 6–7: PCA  →  Z ∈ ℝ^{n_nodes × 2}

In [ ]:
pca = PCA(n_components=2, random_state=0)
Z = pca.fit_transform(H)  # (n_nodes, 2)

print(f'Z shape: {Z.shape}')
print(f'Explained variance ratio: PC1={pca.explained_variance_ratio_[0]:.4f}, '
      f'PC2={pca.explained_variance_ratio_[1]:.4f}')
print(f'Cumulative: {pca.explained_variance_ratio_.sum():.4f}')

## Step 8: Scatter plot — PC1 vs PC2, coloured by grid row

In [ ]:
ROW_COLORS = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a']  # one colour per grid row
ROW_LABELS = ['Row 0 (top)', 'Row 1', 'Row 2', 'Row 3 (bottom)']

fig, ax = plt.subplots(figsize=(9, 7))

# Track which row labels we've added to the legend
legend_added = set()

for i, tau in enumerate(unique_ids):
    word = ID_TO_WORD[tau]
    row, col = WORD_TO_POS[word]
    color = ROW_COLORS[row]
    label = ROW_LABELS[row] if row not in legend_added else None
    legend_added.add(row)

    ax.scatter(Z[i, 0], Z[i, 1], s=120, c=color, edgecolors='k',
               linewidths=0.5, zorder=3, label=label)
    ax.annotate(word, (Z[i, 0], Z[i, 1]),
                textcoords='offset points', xytext=(8, 6),
                fontsize=10, fontweight='bold')

ax.set_xlabel('PC 1', fontsize=12)
ax.set_ylabel('PC 2', fontsize=12)
ax.set_title(
    f'PCA of mean token embeddings (layer 26, Nw={NW})\n'
    f'Var explained: PC1={pca.explained_variance_ratio_[0]:.2%}, '
    f'PC2={pca.explained_variance_ratio_[1]:.2%}',
    fontsize=12
)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

## Bonus: Overlay the 4 × 4 grid layout for reference

In [ ]:
# Build adjacency from the grid
WORD_TO_IDX = {ID_TO_WORD[tau]: i for i, tau in enumerate(unique_ids)}

edges = []
for r in range(4):
    for c in range(4):
        w = GRID[r][c]
        for rr, cc in [(r-1,c),(r+1,c),(r,c-1),(r,c+1)]:
            if 0 <= rr < 4 and 0 <= cc < 4:
                w2 = GRID[rr][cc]
                i1, i2 = WORD_TO_IDX[w], WORD_TO_IDX[w2]
                if i1 < i2:  # avoid duplicate edges
                    edges.append((i1, i2))

fig, ax = plt.subplots(figsize=(9, 7))

# Draw grid edges in PCA space
for i1, i2 in edges:
    ax.plot([Z[i1, 0], Z[i2, 0]], [Z[i1, 1], Z[i2, 1]],
            color='#cccccc', linewidth=1.0, zorder=1)

legend_added = set()
for i, tau in enumerate(unique_ids):
    word = ID_TO_WORD[tau]
    row, col = WORD_TO_POS[word]
    color = ROW_COLORS[row]
    label = ROW_LABELS[row] if row not in legend_added else None
    legend_added.add(row)

    ax.scatter(Z[i, 0], Z[i, 1], s=140, c=color, edgecolors='k',
               linewidths=0.6, zorder=3, label=label)
    ax.annotate(word, (Z[i, 0], Z[i, 1]),
                textcoords='offset points', xytext=(8, 6),
                fontsize=10, fontweight='bold')

ax.set_xlabel('PC 1', fontsize=12)
ax.set_ylabel('PC 2', fontsize=12)
ax.set_title(
    f'PCA of mean token embeddings with grid edges (layer 26, Nw={NW})\n'
    f'Var explained: PC1={pca.explained_variance_ratio_[0]:.2%}, '
    f'PC2={pca.explained_variance_ratio_[1]:.2%}',
    fontsize=12
)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

## Single-file PCA: all 50 timesteps (no averaging)

Load one `.pt` file and scatter every timestep embedding in PC1 vs PC2,
coloured by token id. No pooling / no mean — each of the 50 points is one
raw contextual embedding.

In [ ]:
# ── Pick which batch file to visualise ──
BATCH_IDX = 5  # change to 0‥15 to inspect a different sequence

fpath = EMB_DIR / FILE_PATTERN.format(batch_idx=BATCH_IDX)
assert fpath.exists(), f'Missing file: {fpath}'

save_obj = torch.load(fpath, map_location='cpu', weights_only=False)
single_ids = save_obj[IDS_KEY]
single_emb = save_obj[EMB_KEY]

if single_ids.ndim == 2 and single_ids.shape[0] == 1:
    single_ids = single_ids[0]
if single_emb.ndim == 3 and single_emb.shape[0] == 1:
    single_emb = single_emb[0]

single_ids = single_ids.numpy()
single_emb = single_emb.float().numpy()  # (Nw, d)

# ── PCA on the raw 50 embeddings ──
pca_single = PCA(n_components=2, random_state=0)
Z_single = pca_single.fit_transform(single_emb)  # (Nw, 2)

# ── Compute per-token centroids (for grid edges) ──
unique_single = np.unique(single_ids)
centroids = {}  # tid -> (mean_pc1, mean_pc2)
for tid in unique_single:
    mask = single_ids == tid
    centroids[int(tid)] = Z_single[mask].mean(axis=0)

# ── Build grid edges between tokens present in this file ──
WORD_TO_TID = {w: t for t, w in ID_TO_WORD.items()}
single_edges = []
for r in range(4):
    for c in range(4):
        w1 = GRID[r][c]
        t1 = WORD_TO_TID[w1]
        for rr, cc in [(r-1,c),(r+1,c),(r,c-1),(r,c+1)]:
            if 0 <= rr < 4 and 0 <= cc < 4:
                w2 = GRID[rr][cc]
                t2 = WORD_TO_TID[w2]
                if t1 < t2 and t1 in centroids and t2 in centroids:
                    single_edges.append((t1, t2))

# ── Scatter: one point per timestep, colour = token id ──
cmap = plt.get_cmap('tab20')

fig, ax = plt.subplots(figsize=(9, 7))

# Draw grid edges (centroid-to-centroid)
for t1, t2 in single_edges:
    c1, c2 = centroids[t1], centroids[t2]
    ax.plot([c1[0], c2[0]], [c1[1], c2[1]],
            color='#cccccc', linewidth=1.0, zorder=1)

for idx, tid in enumerate(unique_single):
    mask = single_ids == tid
    word = ID_TO_WORD.get(int(tid), str(tid))
    ax.scatter(
        Z_single[mask, 0], Z_single[mask, 1],
        s=90, alpha=0.85,
        color=cmap(idx % cmap.N),
        edgecolors='k', linewidths=0.4,
        label=word, zorder=3,
    )
    # Label at the centroid
    cx, cy = centroids[int(tid)]
    ax.annotate(word, (cx, cy),
                textcoords='offset points', xytext=(8, 6),
                fontsize=10, fontweight='bold')

ax.set_xlabel('PC 1', fontsize=12)
ax.set_ylabel('PC 2', fontsize=12)
ax.set_title(
    f'Single-file PCA — batch {BATCH_IDX} (layer 26, {len(single_ids)} timesteps)\n'
    f'Var explained: PC1={pca_single.explained_variance_ratio_[0]:.2%}, '
    f'PC2={pca_single.explained_variance_ratio_[1]:.2%}',
    fontsize=12,
)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

## Single random walk — Nw = 500, no batch averaging

One walk of length 1400, window = 500. Each of the 500 timestep embeddings
is plotted as a point coloured by token id, with grid edges connecting
per-token centroids.

In [ ]:
# ── Grid layout for this experiment (from data_generation.py) ──
GRID_OW = [
    ['apple', 'bird',  'car',   'egg'],
    ['house', 'milk',  'plane', 'opera'],
    ['box',   'sand',  'sun',   'mango'],
    ['rock',  'math',  'code',  'phone'],
]
WORD_TO_POS_OW = {}
for r, row in enumerate(GRID_OW):
    for c, w in enumerate(row):
        WORD_TO_POS_OW[w] = (r, c)

# ── Load the single-walk file ──
ONE_WALK_PATH = EMB_DIR / 'one_walk/vdestasio/reprs_paper_grid_one_rw_1400_b0000_layer26.pt'
NW_ONE = 200  #  window


assert ONE_WALK_PATH.exists(), f'Missing file: {ONE_WALK_PATH}'

ow_obj = torch.load(ONE_WALK_PATH, map_location='cpu', weights_only=False)
ow_ids = ow_obj[IDS_KEY]
ow_emb = ow_obj[EMB_KEY]

if ow_ids.ndim == 2 and ow_ids.shape[0] == 1:
    ow_ids = ow_ids[0]
if ow_emb.ndim == 3 and ow_emb.shape[0] == 1:
    ow_emb = ow_emb[0]

# Take the recent window W = min(NW_ONE, L)
L = ow_ids.shape[0]
W = min(NW_ONE, L)
ow_ids = ow_ids[-W:].numpy()
ow_emb = ow_emb[-W:].float().numpy()  # (W, d)

print(f'Window: {W} timesteps  (file length {L})')
print(f'Embedding dim: {ow_emb.shape[1]}')
print(f'Unique tokens: {len(np.unique(ow_ids))}')

# ── PCA on raw embeddings ──
pca_ow = PCA(n_components=2, random_state=0)
Z_ow = pca_ow.fit_transform(ow_emb)  # (W, 2)

# ── Per-token centroids & grid edges ──
unique_ow = np.unique(ow_ids)
centroids_ow = {}
for tid in unique_ow:
    mask = ow_ids == tid
    centroids_ow[int(tid)] = Z_ow[mask].mean(axis=0)

WORD_TO_TID_OW = {w: t for t, w in ID_TO_WORD.items()}
edges_ow = []
for r in range(4):
    for c in range(4):
        w1 = GRID_OW[r][c]
        t1 = WORD_TO_TID_OW[w1]
        for rr, cc in [(r-1,c),(r+1,c),(r,c-1),(r,c+1)]:
            if 0 <= rr < 4 and 0 <= cc < 4:
                w2 = GRID_OW[rr][cc]
                t2 = WORD_TO_TID_OW[w2]
                if t1 < t2 and t1 in centroids_ow and t2 in centroids_ow:
                    edges_ow.append((t1, t2))

# ── Plot ──
cmap_ow = plt.get_cmap('tab20')
fig, ax = plt.subplots(figsize=(9, 7))

# Grid edges (centroid-to-centroid)
for t1, t2 in edges_ow:
    c1, c2 = centroids_ow[t1], centroids_ow[t2]
    ax.plot([c1[0], c2[0]], [c1[1], c2[1]],
            color='#cccccc', linewidth=1.0, zorder=1)

ROW_COLORS_OW = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a']
ROW_LABELS_OW = ['Row 0: apple bird car egg',
                 'Row 1: house milk plane opera',
                 'Row 2: box sand sun mango',
                 'Row 3: rock math code phone']
legend_added_ow = set()

for idx, tid in enumerate(unique_ow):
    mask = ow_ids == tid
    word = ID_TO_WORD.get(int(tid), str(tid))
    row_grid, _ = WORD_TO_POS_OW.get(word, (-1, -1))
    color = ROW_COLORS_OW[row_grid] if row_grid >= 0 else cmap_ow(idx % cmap_ow.N)
    label = ROW_LABELS_OW[row_grid] if row_grid >= 0 and row_grid not in legend_added_ow else None
    if row_grid >= 0:
        legend_added_ow.add(row_grid)
    ax.scatter(
        Z_ow[mask, 0], Z_ow[mask, 1],
        s=60, alpha=0.75,
        color=color,
        edgecolors='k', linewidths=0.3,
        label=label,
        zorder=3,
    )
    # Label at centroid
    cx, cy = centroids_ow[int(tid)]
    ax.annotate(word, (cx, cy),
                textcoords='offset points', xytext=(8, 6),
                fontsize=9, fontweight='bold')

ax.set_xlabel('PC 1', fontsize=12)
ax.set_ylabel('PC 2', fontsize=12)
ax.set_title(
    f'Single random walk — PCA of {W} timesteps (layer 26, no averaging)\n'
    f'Var explained: PC1={pca_ow.explained_variance_ratio_[0]:.2%}, '
    f'PC2={pca_ow.explained_variance_ratio_[1]:.2%}',
    fontsize=12,
)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9,
          framealpha=0.9, title='token (count)')
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

## Single walk — last activation per token (no window, no averaging)

For each of the 16 grid tokens, take the embedding from the **last timestep**
where that token appeared in the sequence. This gives exactly one embedding
per token → H ∈ ℝ^{16 × d}, then PCA to 2D.

In [ ]:
# ── Load full sequence (no windowing) ──
LAST_ACT_PATH = EMB_DIR / 'one_walk/vdestasio/reprs_paper_grid_one_rw_1400_b0000_layer26.pt'
assert LAST_ACT_PATH.exists(), f'Missing: {LAST_ACT_PATH}'

la_obj = torch.load(LAST_ACT_PATH, map_location='cpu', weights_only=False)
la_ids = la_obj[IDS_KEY]
la_emb = la_obj[EMB_KEY]

if la_ids.ndim == 2 and la_ids.shape[0] == 1:
    la_ids = la_ids[0]
if la_emb.ndim == 3 and la_emb.shape[0] == 1:
    la_emb = la_emb[0]

la_ids_np = la_ids.numpy()
la_emb_np = la_emb.float().numpy()  # (L, d)

print(f'Sequence length: {len(la_ids_np)},  embedding dim: {la_emb_np.shape[1]}')

# ── For each token, pick the embedding at its LAST occurrence ──
unique_tids = sorted(ID_TO_WORD.keys())
n_nodes_la = len(unique_tids)
H_last = np.zeros((n_nodes_la, la_emb_np.shape[1]), dtype=np.float32)
last_positions = {}

for i, tau in enumerate(unique_tids):
    positions = np.where(la_ids_np == tau)[0]
    assert len(positions) > 0, (
        f'Token {tau} ({ID_TO_WORD[tau]}) never appears in the sequence.'
    )
    last_t = positions[-1]          # last timestep for this token
    H_last[i] = la_emb_np[last_t]
    last_positions[tau] = int(last_t)

print(f'\nH_last shape: {H_last.shape}  (one embedding per token)')
print('Last-occurrence timesteps:')
for tau in unique_tids:
    print(f'  {ID_TO_WORD[tau]:>8s} (id {tau:>6d}):  t = {last_positions[tau]}')

# ── PCA ──
pca_la = PCA(n_components=2, random_state=0)
Z_la = pca_la.fit_transform(H_last)  # (16, 2)

# ── Grid edges (using GRID_OW from data_generation.py) ──
WORD_TO_TID_LA = {w: t for t, w in ID_TO_WORD.items()}
edges_la = []
for r in range(4):
    for c in range(4):
        w1 = GRID_OW[r][c]
        t1 = WORD_TO_TID_LA[w1]
        for rr, cc in [(r-1,c),(r+1,c),(r,c-1),(r,c+1)]:
            if 0 <= rr < 4 and 0 <= cc < 4:
                w2 = GRID_OW[rr][cc]
                t2 = WORD_TO_TID_LA[w2]
                if t1 < t2:
                    edges_la.append((t1, t2))

# Map tid → index in unique_tids for Z_la rows
tid_to_row = {tau: i for i, tau in enumerate(unique_tids)}

# ── Plot ──
ROW_COLORS_LA = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a']
ROW_LABELS_LA = ['Row 0: apple bird car egg',
                 'Row 1: house milk plane opera',
                 'Row 2: box sand sun mango',
                 'Row 3: rock math code phone']

fig, ax = plt.subplots(figsize=(9, 7))

# Grid edges
for t1, t2 in edges_la:
    i1, i2 = tid_to_row[t1], tid_to_row[t2]
    ax.plot([Z_la[i1, 0], Z_la[i2, 0]], [Z_la[i1, 1], Z_la[i2, 1]],
            color='#cccccc', linewidth=1.0, zorder=1)

legend_added_la = set()
for i, tau in enumerate(unique_tids):
    word = ID_TO_WORD[tau]
    row_g, _ = WORD_TO_POS_OW.get(word, (-1, -1))
    color = ROW_COLORS_LA[row_g] if row_g >= 0 else 'grey'
    label = ROW_LABELS_LA[row_g] if row_g >= 0 and row_g not in legend_added_la else None
    if row_g >= 0:
        legend_added_la.add(row_g)

    ax.scatter(Z_la[i, 0], Z_la[i, 1], s=140, c=color,
               edgecolors='k', linewidths=0.5, zorder=3, label=label)
    ax.annotate(f'{word}  (t={last_positions[tau]})',
                (Z_la[i, 0], Z_la[i, 1]),
                textcoords='offset points', xytext=(8, 6),
                fontsize=9, fontweight='bold')

ax.set_xlabel('PC 1', fontsize=12)
ax.set_ylabel('PC 2', fontsize=12)
ax.set_title(
    f'Last-activation PCA — one embedding per token (layer 26)\n'
    f'Var explained: PC1={pca_la.explained_variance_ratio_[0]:.2%}, '
    f'PC2={pca_la.explained_variance_ratio_[1]:.2%}',
    fontsize=12,
)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()